In [1]:
# ============================================
# Input Validation & Error Handling
# Validates all inputs before feeding to
# any of the 3 models (KNN, DT, SVM)
# ============================================

import pandas as pd
import numpy as np

FEATURE_NAMES = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

# Valid ranges based on Iris dataset
FEATURE_RANGES = {
    'SepalLengthCm': (4.0, 8.0),
    'SepalWidthCm':  (2.0, 5.0),
    'PetalLengthCm': (1.0, 7.0),
    'PetalWidthCm':  (0.1, 2.6),
}

def validate_input(input_data):
    """
    Validates flower measurement input.
    Returns (True, cleaned_df) if valid,
    or (False, error_message) if invalid.
    """
    # Check 1: Empty input
    if input_data is None or (hasattr(input_data, '__len__') and len(input_data) == 0):
        return False, "❌ Error: Input is empty. Please provide 4 measurements."

    # Check 2: Correct number of features
    if len(input_data) != 4:
        return False, f"❌ Error: Expected 4 features, got {len(input_data)}."

    # Check 3: All values must be numbers
    for i, val in enumerate(input_data):
        if not isinstance(val, (int, float, np.floating)):
            return False, f"❌ Error: Feature '{FEATURE_NAMES[i]}' must be a number, got '{type(val).__name__}'."

    # Check 4: No NaN or infinite values
    for i, val in enumerate(input_data):
        if np.isnan(val) or np.isinf(val):
            return False, f"❌ Error: Feature '{FEATURE_NAMES[i]}' has invalid value ({val})."

    # Check 5: Values within realistic range
    for i, val in enumerate(input_data):
        low, high = FEATURE_RANGES[FEATURE_NAMES[i]]
        if not (low <= val <= high):
            return False, f"❌ Error: '{FEATURE_NAMES[i]}' value {val} is out of range [{low}, {high}]."

    # All checks passed — return clean DataFrame
    df_input = pd.DataFrame([input_data], columns=FEATURE_NAMES)
    return True, df_input


def predict_with_validation(model, input_data, model_name="Model"):
    """
    Safely predicts with full validation.
    """
    is_valid, result = validate_input(input_data)
    if not is_valid:
        print(f"[{model_name}] {result}")
        return None
    prediction = model.predict(result)[0]
    print(f"[{model_name}] ✅ Prediction: {prediction}")
    return prediction

In [2]:
# Load and train models for testing
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/iris_cleaned.csv')
X = df[FEATURE_NAMES]
y = df['Species']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)

print("=" * 50)
print("TESTING INVALID INPUTS")
print("=" * 50)

predict_with_validation(knn, [],                          "KNN")  # empty
predict_with_validation(knn, [5.1, 3.5],                  "KNN")  # too few
predict_with_validation(knn, [5.1, 3.5, "abc", 0.2],     "KNN")  # wrong type
predict_with_validation(knn, [5.1, 3.5, float('nan'), 0.2], "KNN")  # NaN
predict_with_validation(knn, [99, 3.5, 1.4, 0.2],        "KNN")  # out of range

print("\n" + "=" * 50)
print("TESTING VALID INPUTS")
print("=" * 50)
predict_with_validation(knn, [5.1, 3.5, 1.4, 0.2], "KNN")
predict_with_validation(knn, [6.2, 2.9, 4.3, 1.3], "KNN")
predict_with_validation(knn, [7.3, 2.9, 6.3, 1.8], "KNN")

TESTING INVALID INPUTS
[KNN] ❌ Error: Input is empty. Please provide 4 measurements.
[KNN] ❌ Error: Expected 4 features, got 2.
[KNN] ❌ Error: Feature 'PetalLengthCm' must be a number, got 'str'.
[KNN] ❌ Error: Feature 'PetalLengthCm' has invalid value (nan).
[KNN] ❌ Error: 'SepalLengthCm' value 99 is out of range [4.0, 8.0].

TESTING VALID INPUTS
[KNN] ✅ Prediction: Iris-setosa
[KNN] ✅ Prediction: Iris-versicolor
[KNN] ✅ Prediction: Iris-virginica


'Iris-virginica'